<a href="https://colab.research.google.com/github/daviminati/Trab_Final_IA/blob/main/Sprint_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sprint 4: Implementando um Modelo GPT do Zero

**Disciplina / Projeto Integrador LLM**

Neste notebook, construímos a arquitetura completa do modelo GPT conforme o Capítulo 4 do livro *Build a Large Language Model (From Scratch)* de Sebastian Raschka.

---

## 4.1 Configuração Inicial e Importações

Começamos importando as bibliotecas essenciais do PyTorch e verificando as versões do ambiente de execução.

In [1]:
# 1. Configuração do ambiente e importações essenciais
from importlib.metadata import version
import math
import torch
import torch.nn as nn

print("torch version:", version("torch"))

torch version: 2.11.0+cpu


## 4.2 Entregável 2: Normalização de Camada (LayerNorm)

A **Layer Normalization** ajusta as ativações de cada camada para que tenham média 0 e variância 1 ao longo da dimensão de embedding.

Diferente do BatchNorm, a LayerNorm opera individualmente em cada amostra/token, estabilizando o fluxo de gradientes durante o treinamento de redes profundas como o GPT.

Nesta célula implementamos:
* A subtração da média (`mean`) e divisão pelo desvio padrão (`sqrt(var + eps)`).
* Parâmetros aprendíveis de escala (`scale` / $\gamma$) e deslocamento (`shift` / $\beta$).

In [2]:
# Implementação da classe LayerNorm
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift

# Teste básico da LayerNorm com um tensor aleatório
torch.manual_seed(123)
exemplo_batch = torch.randn(2, 3, 768) # Batch=2, Tokens=3, Dim=768
ln = LayerNorm(768)
out_ln = ln(exemplo_batch)

print("Média após LayerNorm (próxima de 0):", out_ln.mean(dim=-1).detach().numpy().round(2))
print("Variância após LayerNorm (próxima de 1):", out_ln.var(dim=-1, unbiased=False).detach().numpy().round(2))

Média após LayerNorm (próxima de 0): [[ 0.  0. -0.]
 [ 0.  0.  0.]]
Variância após LayerNorm (próxima de 1): [[1. 1. 1.]
 [1. 1. 1.]]


## 4.3 Função de Ativação GELU

Em vez da ReLU tradicional, modelos da família GPT utilizam a função de ativação **GELU** (Gaussian Error Linear Unit).

A GELU é uma função suave e não linear que pondera os valores de entrada por sua probabilidade sob uma distribuição gaussiana. Ela permite pequenos gradientes negativos para entradas abaixo de zero, o que melhora a convergência e a expressividade do modelo.

In [3]:
# Implementação da ativação GELU (Aproximação utilizada no GPT-2)
class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1.0 + torch.tanh(
            math.sqrt(2.0 / math.pi) * (x + 0.044715 * torch.pow(x, 3))
        ))

# Validação do comportamento da GELU
gelu = GELU()
x_teste = torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0])
print("Valores de entrada: ", x_teste)
print("Saída após a GELU:", gelu(x_teste))

Valores de entrada:  tensor([-2., -1.,  0.,  1.,  2.])
Saída após a GELU: tensor([-0.0454, -0.1588,  0.0000,  0.8412,  1.9546])


## 4.4 Rede Feed-Forward (FFN)

A **Feed-Forward Network** do Transformer consiste em duas camadas lineares com a ativação GELU entre elas:
1. **Primeira projeção:** Expande a dimensão do embedding por um fator de 4 ($4 \times d_{out}$).
2. **Ativação:** Aplica a função GELU na dimensão expandida.
3. **Segunda projeção:** Projeta o tensor de volta para a dimensão original ($d_{out}$).

Essa expansão temporária permite que a rede aprenda representações e combinações de recursos mais ricas em cada posição da sequência.

In [4]:
# Implementação da subclasse FeedForward
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"])
        )

    def forward(self, x):
        return self.layers(x)

# Configuração de teste para a FFN
cfg_teste = {"emb_dim": 768}
ffn = FeedForward(cfg_teste)
out_ffn = ffn(exemplo_batch)

print("Shape de entrada na FFN:", exemplo_batch.shape)
print("Shape de saída da FFN:  ", out_ffn.shape)

Shape de entrada na FFN: torch.Size([2, 3, 768])
Shape de saída da FFN:   torch.Size([2, 3, 768])


## 4.5 Módulo Multi-Head Attention

Reutilizamos aqui a classe `MultiHeadAttention` com máscara causal desenvolvida na Sprint 3. Ela permite que cada token preste atenção a tokens anteriores em múltiplos subespaços de representação em paralelo.

In [10]:
# Implementação da classe MultiHeadAttention (da Sprint 3)
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out precisa ser divisível por num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)

        self.register_buffer(
            'mask',
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        keys    = self.W_key(x)
        queries = self.W_query(x)
        values  = self.W_value(x)

        keys    = keys.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        values  = values.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)

        attn_scores = queries @ keys.transpose(2, 3)
        mask_bool = self.mask[:num_tokens, :num_tokens].bool()

        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / (keys.shape[-1]**0.5), dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vec = (attn_weights @ values).transpose(1, 2)
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        return self.out_proj(context_vec)